<a href="https://colab.research.google.com/github/kev-aaaaa/dataSciPortfolio/blob/main/Automated_Scholarhsip_Eligibility_Filtration_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Suppose a scholarship exists where it has the following conditions:

1. QWA Requirements vary by year level:


*   1st-year scholars must uphold a GWA of atleast 2.500 cumulatively for the year or lower (and vice versa if 5.00 is considered the pinnacle of excellence in certain universities)


*   2nd-year scholars must uphold a GWA of 2.000 or below each term (semestral, trimestral, quarterm)

* 3rd-year to graduating scholars must not have atleast one failing course throughout the rest of the year level.

2. Scholars must not have any deficiencies (failing grades, withdrawn courses, and incomplete courses)



This project focuses on filtering data from a general university population (by their student numbers) into two outputting two separate .csv files filtering the concerned scholars by:

1. Scholarship eligibility
2. Year level

Stored in a Pandas dataframe for easy data processing, and readability across concerned faculty.



The code block below is a function **that simulates the possible dataset of a university's records of the general student population.**

This is done to avoid using real-world university data.



In [1]:
import random as rnd
import pandas as pd

# Header and base formatting for scholar records
headers = ["XYZ UNIVERSTY STUDENT POPULATION AUDIT: 3rd Trimester AY 2025-2026", "=== XYZ Government Organization Scholarship Report ===\n\n"]

rawRecords = "3T_AY_25-26_RAW_RECORDS.csv"
scholarRecords = "GovernmentScholarRecords.csv"
finalRecords = "filteredScholarRecords.csv"

def registrarFiles():
  ''' Creates 4500 (general popualation) fictional records of random students with expected output in a Pandas DataFrame.

      Student number: {YYYY}{XXXXXX} | Year Level | QWA | Failed Courses | Withdrawn Courses | Incomplete Courses | Scholarship | Units Enrolled

      Its only purpose is to serve as testing grounds for code functionality and debugging.
  '''
  with open(rawRecords, 'w+') as file:

    recordsDict = {"Student Number": [], "TWA": [], "Year Level": [], "Failed Courses": [], "Withdrawn Courses": [], "Incomplete Courses": [], "Scholarship": [], "Units Enrolled": []}

    # Creates dictionary entries for pd Dataframe through a while loop
    i = 0
    while (i != 2500):

      # All other parameters
      studentTWA = rnd.uniform(1.00, 5.00)
      failedCourses = rnd.randint(0, 1)
      withdrawnCourses = rnd.randint(0,1)
      incompleteCourses = rnd.randint(0,1)
      units = rnd.randint(12, 21)

      #Random selection from choice of scholarships
      scholarships = ("NONE", "ACADEMIC", "GOVERNMENT", "VARSITY")
      studentScholarships = rnd.choice(scholarships)

      #Student Number
      studentNo = f"{rnd.randint(2022,2026)}{rnd.randint(100000,999999)}"
      recordsDict["Student Number"].append(studentNo)

      # Year level is based on the year on the student number {YYYY}{XXXXXX}. QWA Requirements for eligibility differ by year.
      # This is important as no DOST-SEI scholar is expected to fail, withdraw, or have incomplete courses.
      # Incoming first years for 1Q AY 26-27 will be falsely flagged as failing without this system.


      if ("2026" in studentNo):
        recordsDict["Year Level"].append(1)
      elif ("2025" in studentNo):
        recordsDict["Year Level"].append(2)
      elif ("2024" in studentNo):
        recordsDict["Year Level"].append(3)
      elif ("2023" in studentNo):
        recordsDict["Year Level"].append(4)
      else:
        recordsDict["Year Level"].append(5)

      recordsDict["Scholarship"].append(studentScholarships)
      recordsDict["Units Enrolled"].append(units)

     # Context indicates that students with "2026" on their student numbers are new enrollees. This prevents the software from loading falsified academic data (except for scholarship and units)in the new quarterm.
      if ("2026" in studentNo):
        recordsDict["TWA"].append(0)
        recordsDict["Failed Courses"].append(0)
        recordsDict["Withdrawn Courses"].append(0)
        recordsDict["Incomplete Courses"].append(0)
      else:
        recordsDict["TWA"].append(studentTWA)
        recordsDict["Failed Courses"].append(failedCourses)
        recordsDict["Withdrawn Courses"].append(withdrawnCourses)
        recordsDict["Incomplete Courses"].append(incompleteCourses)

      i += 1

    studentDF = pd.DataFrame(recordsDict)
    studentDF.to_csv(rawRecords, index=False)

    file.seek(0)

  print(f"4500 fictional student records (unorganized) generated for data filtration and Government scholarship eligibility.")

registrarFiles()

# Uncomment to check raw database
print(headers[0])
pd.read_csv(rawRecords)





2500 fictional student records (unorganized) generated for data filtration and DOST-SEI scholarship eligibility.
XYZ UNIVERSTY STUDENT POPULATION AUDIT: 3rd Trimester AY 2025-2026


,Student Number,TWA,Year Level,Failed Courses,Withdrawn Courses,Incomplete Courses,Scholarship,Units Enrolled
0,2025514116,3.716941,2,0,0,1,VARSITY,14
1,2025859830,1.575169,2,0,1,0,VARSITY,19
2,2025632224,2.494964,2,0,1,0,GOVERNMENT,13
3,2022968220,4.090536,5,0,1,0,VARSITY,20
4,2022531728,2.485117,5,1,0,1,VARSITY,21
...,...,...,...,...,...,...,...,...
2495,2026985402,0.000000,1,0,0,0,GOVERNMENT,18
2496,2023665159,4.754265,4,0,0,0,VARSITY,15
2497,2022895667,2.218068,5,0,1,1,ACADEMIC,14
2498,2026438894,0.000000,1,0,0,0,NONE,12


This function effectively filters the designated dataset above based on the specified eligibility requirements to continue enjoying the scholarship.

**To change the target scholarship, kindly change the code line:**


```
 scholarDF = rawDF[rawDF.iloc[:, 6] == "Government"].copy()
```
Into a target scholarship of choice, assuming same initial conditions.


In [10]:
def ScholarValidation(csv_file):
  ''' This function filters the raw student data from the whole population and does the following functions:

    1. Filters all students part of the Government scholarship program (regardless of type)
    2. Checks each scholar's eligibility based on the following parameters:
    Year 1: None (new enrollees given context: 1Q 2026-2027)
    Year 2: QWA of 2.5 (upcoming 2nd year student)
    Year 3 - 4: QWA of 3.00 (expected NO failing/incomplete/withdrawal of courses)

    3. Presents all these findings with the following Pandas Dataframe format, appending to the original file. The student number serves as the index for easy lookup:
    YEAR LEVEL | WITH DEFICIENCIES | RECOMMENDED ACTION | SCHOLARSHIP STATUS

    4. Returns two files:
      1) Organized by SCHOLARSHIP STATUS, and;
      2) Organized by YEAR LEVEL.
  '''
  rawDF = pd.read_csv(rawRecords)

  # Immediately copies all the DOST-SEI students into a scholarDF, an identical dataframe.
  scholarDF = rawDF[rawDF.iloc[:, 6] == "GOVERNMENT"].copy()

  # In-built Pandas infrastructure to assign values and set restrictions
  scholarDF["TWARequirement"] = 3.00
  scholarDF.loc[scholarDF.iloc[:, 2] == 1, 'TWARequirement'] = 0.00
  scholarDF.loc[scholarDF.iloc[:, 2] == 2, 'TWARequirenent'] = 2.500

  #Test for eligibility conditions (fail is default)
  eligible = ((scholarDF.iloc[:, 3] == 0) & (scholarDF.iloc[:, 4] == 0) & (scholarDF.iloc[:, 5] == 0) & (scholarDF.iloc[:, 1] <= scholarDF['TWARequirement']))

  scholarDF["WITH DEFICIENCIES"] = "YES"
  scholarDF["RECOMMENDED ACTION"] = 'REQUEST LETTER FOR APPEAL'
  scholarDF["SCHOLARSHIP STATUS"] = ' TERMINATED '

  #Automatically overrides columns for students that are actually eligible
  scholarDF.loc[eligible, "WITH DEFICIENCIES"] = 'NO'
  scholarDF.loc[eligible, "RECOMMENDED ACTION"] = "SUBMIT SCHOLAR'S PROGRESS REPORT FORM"
  scholarDF.loc[eligible, "SCHOLARSHIP STATUS"] = 'ACTIVE'

  scholarDF = scholarDF.sort_values(by="SCHOLARSHIP STATUS")
  scholarDF = scholarDF.set_index('Student Number')

  scholarDF.to_csv(scholarRecords)

  yearLevelDF = scholarDF.copy()
  yearLevelDF = yearLevelDF.sort_values(by='Year Level')

  yearLevelDF.to_csv(finalRecords)


ScholarValidation(rawRecords)



**Access the filtered list of scholars by eligibility**

In [11]:
pd.read_csv(scholarRecords)

,Student Number,TWA,Year Level,Failed Courses,Withdrawn Courses,Incomplete Courses,Scholarship,Units Enrolled,TWARequirement,TWARequirenent,WITH DEFICIENCIES,RECOMMENDED ACTION,SCHOLARSHIP STATUS
0,2025632224,2.494964,2,0,1,0,GOVERNMENT,13,3.0,2.5,YES,REQUEST LETTER FOR APPEAL,TERMINATED
1,2025638255,2.974222,2,1,0,1,GOVERNMENT,15,3.0,2.5,YES,REQUEST LETTER FOR APPEAL,TERMINATED
2,2025606317,4.682577,2,1,0,0,GOVERNMENT,16,3.0,2.5,YES,REQUEST LETTER FOR APPEAL,TERMINATED
3,2022348875,2.295628,5,1,1,1,GOVERNMENT,16,3.0,NaN,YES,REQUEST LETTER FOR APPEAL,TERMINATED
4,2025473620,2.078071,2,0,0,1,GOVERNMENT,17,3.0,2.5,YES,REQUEST LETTER FOR APPEAL,TERMINATED
...,...,...,...,...,...,...,...,...,...,...,...,...,...
594,2023862352,2.970097,4,0,0,0,GOVERNMENT,13,3.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
595,2026912008,0.000000,1,0,0,0,GOVERNMENT,12,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
596,2026847822,0.000000,1,0,0,0,GOVERNMENT,17,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
597,2026119262,0.000000,1,0,0,0,GOVERNMENT,18,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE


**Accesses the filtered list of scholars by year level**

In [12]:
pd.read_csv(finalRecords)

,Student Number,TWA,Year Level,Failed Courses,Withdrawn Courses,Incomplete Courses,Scholarship,Units Enrolled,TWARequirement,TWARequirenent,WITH DEFICIENCIES,RECOMMENDED ACTION,SCHOLARSHIP STATUS
0,2026634070,0.000000,1,0,0,0,GOVERNMENT,19,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
1,2026912008,0.000000,1,0,0,0,GOVERNMENT,12,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
2,2026847822,0.000000,1,0,0,0,GOVERNMENT,17,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
3,2026119262,0.000000,1,0,0,0,GOVERNMENT,18,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
4,2026662738,0.000000,1,0,0,0,GOVERNMENT,17,0.0,NaN,NO,SUBMIT SCHOLAR'S PROGRESS REPORT FORM,ACTIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...
594,2022286823,3.804814,5,1,1,0,GOVERNMENT,19,3.0,NaN,YES,REQUEST LETTER FOR APPEAL,TERMINATED
595,2022743570,4.402995,5,0,1,1,GOVERNMENT,21,3.0,NaN,YES,REQUEST LETTER FOR APPEAL,TERMINATED
596,2022894212,2.813622,5,1,0,1,GOVERNMENT,14,3.0,NaN,YES,REQUEST LETTER FOR APPEAL,TERMINATED
597,2022475673,3.908724,5,0,1,1,GOVERNMENT,18,3.0,NaN,YES,REQUEST LETTER FOR APPEAL,TERMINATED
